# Lezione 11. Architetture degli agenti: da A3C a IMPALA

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

Benvenuta o benvenuto alla lezione 11. Nella lezione precedente hai visto come Rainbow combina diversi miglioramenti sulla Rete Q Profonda (DQN, dall'inglese Deep Q-Network) e come la prospettiva distribuzionale cambia il modo di stimare il valore di un'azione. Tutto questo risponde a una domanda: come far sì che un singolo agente impari meglio? Oggi rispondi a una domanda diversa e, per gli ambienti complessi su larga scala, altrettanto importante: come far sì che molti agenti imparino più in fretta, insieme?

Pensa ai casi d'uso che vedrai nella lezione 12, l'ultima del corso: sistemi capaci di giocare a Go, scacchi o StarCraft a un livello sovrumano. Nessuno di questi è stato addestrato con un singolo agente su una sola macchina per settimane. Sono stati addestrati con centinaia o migliaia di copie dell'agente che giocano in parallelo, accumulando esperienza a una velocità che nessuna macchina isolata potrebbe eguagliare. Oggi impari questa architettura: prima l'Actor-Critic di Vantaggio Asincrono (A3C, dall'inglese Asynchronous Advantage Actor-Critic), che sostituisce la memoria di replay con il parallelismo, e poi IMPALA, che porta la stessa idea da un singolo computer a un intero cluster.

## Scaletta della lezione

1. Il collo di bottiglia che la Rete Q Profonda lascia alle spalle
2. A3C: sostituisce la memoria con il parallelismo
3. Consigli pratici di ingegneria
4. IMPALA: da un computer a un cluster, con V-trace
5. Codice dal vivo: "Catch" con un actor-critic semplice
6. Punti chiave e cosa ti aspetta nella lezione 12


## 1. Il collo di bottiglia che la Rete Q Profonda lascia alle spalle

*Traccia della lezione: prima di scrivere qualsiasi cosa alla lavagna, chiedi alla classe quali due elementi ha introdotto DQN per poter apprendere in modo stabile (memoria di replay e rete target) e perché servivano. Lascia rispondere prima di proseguire.*

Ricorda i due problemi di correlazione che hanno costretto DQN a introdurre una memoria di replay e una rete target:

- I dati generati da un agente che gioca in modo continuo sono molto correlati tra passi consecutivi
- Una rete neurale, per la sua naturale tendenza a produrre funzioni regolari, assegna valori simili a stati simili, il che fa sì che l'obiettivo di addestramento insegua sé stesso

La memoria di replay risolve entrambi i problemi conservando fino a un milione di fotogrammi e campionando lotti casuali. Ma in cambio introduce una dipendenza stretta dall'apprendimento fuori politica e lascia inutilizzata gran parte della capacità di una scheda grafica (GPU): le reti piccole che funzionano bene in questo schema non riescono a saturarne le prestazioni, quindi addestrare un solo gioco di Atari poteva richiedere fino a una settimana su una sola GPU.

**La domanda che ha aperto la strada ad A3C è stata semplice: se il problema di fondo è la correlazione dei dati, serve memorizzare l'esperienza passata per spezzarla, oppure basta generare esperienza diversificata nel momento?**


## 2. A3C: sostituisce la memoria con il parallelismo

*Traccia della lezione: racconta l'analogia ad alta voce, senza ancora mostrare alcuno schema. È il pezzo che più aiuta l'idea a "fare clic".*

Immagina che, invece di una sola persona che gioca una partita dopo l'altra e annota ogni mossa su un quaderno per ripassarla più tardi, ci siano 16 persone che giocano partite diverse nello stesso momento, ciascuna sulla propria scacchiera. Ogni tanto, ciascun giocatore consulta la strategia comune più recente, gioca qualche turno con essa, calcola quale aggiustamento la migliorerebbe e lo sussurra a un quaderno condiviso, senza aspettare il proprio turno né chiedere il permesso alle altre. Poiché tutti giocano situazioni diverse nello stesso momento, nessun singolo aggiustamento trascina il quaderno comune verso un angolo distorto: la diversità simultanea tra i giocatori svolge lo stesso ruolo che prima svolgeva la diversità temporale della memoria di replay, ma senza dover memorizzare nulla.

Così funziona A3C. Ogni lavoratore —un thread di processo con una propria copia dell'ambiente— ripete questo ciclo:

1. Prende i parametri più recenti della rete condivisa
2. Gioca un numero fisso di passi con la politica attuale
3. Calcola un gradiente e lo applica in modo asincrono a quella rete condivisa, senza bloccare gli altri lavoratori né sincronizzarsi con loro

Non ci sono memoria di replay né rete target: la stessa concorrenza di lavoratori che esplorano punti diversi dell'ambiente stabilizza l'apprendimento.

### L'algoritmo dentro ogni lavoratore: A2C

La componente che impara dentro ogni lavoratore è un Actor-Critic di Vantaggio (A2C, dall'inglese Advantage Actor-Critic), la stessa famiglia che hai studiato nella lezione nove: una rete che prevede, allo stesso tempo, una distribuzione di probabilità sulle azioni (l'attore) e una stima del valore dello stato (il critico).

L'attore tronca il ritorno ai successivi n passi, usa il critico per completare il resto tramite *bootstrapping* e sottrae il valore dello stato attuale come riferimento; a questa differenza si dà il nome di **vantaggio**, ed è ciò che moltiplica il gradiente della politica. C'è un motivo ulteriore per cui questo schema è più stabile che regolare valori Q isolati: l'output dell'attore è una distribuzione che somma a uno, quindi aumentare la probabilità di un'azione riduce automaticamente quella delle altre. È una forma di stabilizzazione numerica già incorporata nel progetto stesso, che un valore Q senza vincoli non possiede.

### Cosa mostrano i numeri

*Traccia della lezione: chiedi alla classe di prevedere, prima di mostrare la cifra, se A3C scalerà meglio o peggio delle varianti a un solo passo passando da un thread a 16.*

Gli esperimenti di scalabilità con 16 core di una unità di elaborazione centrale (CPU) e 16 lavoratori, rispetto a DQN con GPU, su sette giochi di Atari, danno cifre concrete:

- L'actor-critic è scalato in modo sublineare: tra 12 e 13 volte più veloce con 16 thread, rispetto alle 16 volte ideali
- Le varianti a un passo sono scalate in modo superlineare, circa 20 volte più veloci, grazie all'effetto stabilizzante di avere più lavoratori che esplorano contemporaneamente
- In Breakout, un solo lavoratore con 50 milioni di fotogrammi arrivava a circa 250 punti; gli stessi 50 milioni di fotogrammi distribuiti su 16 lavoratori arrivavano a circa 300 punti, per l'effetto di mediare su una porzione più ampia dello spazio degli stati
- Nonostante le varianti a un passo scalassero meglio in velocità pura, A3C restava nel complesso l'opzione più efficiente sui dati

Il risultato pratico: A3C eguagliava o superava DQN su Atari in un solo giorno, su una sola macchina, senza alcuna GPU; con quattro giorni di addestramento migliorava ancora di più. Funziona sia con reti ricorrenti sia con reti feed-forward, è stato uno dei primi metodi capaci di imparare a navigare labirinti tridimensionali casuali a partire dai pixel, e si adatta al controllo continuo sostituendo la distribuzione sulle azioni discrete con la media e la varianza di una distribuzione gaussiana.

**Ha anche limiti espliciti**: può essere meno stabile di DQN, a volte richiede di regolare gli iperparametri gioco per gioco, sfrutta peggio la GPU e, soprattutto, resta legato ai thread di un'unica macchina. Quest'ultimo limite è proprio quello che risolve IMPALA.


## 3. Consigli pratici di ingegneria

*Traccia della lezione: presenta questa sezione come il ponte verso il codice dal vivo. Insisti sul fatto che "debugga prima in piccolo" non è un suggerimento facoltativo: è il criterio che applicherai tra poco con "Catch".*

Prima di scalare verso qualcosa come A3C conviene interiorizzare un principio di lavoro, costi quel che costi: debugga prima in piccolo.

- **Inizia con un problema giocattolo**: scegli un ambiente minimo che condivida le caratteristiche chiave del problema reale (attribuzione del credito a lungo termine, dimensione dello spazio di input) ma che tu possa risolvere in pochi minuti sul tuo portatile. L'esempio consigliato è "Catch": una palla cade dall'alto su una piccola griglia e controlli una racchetta che puoi spostare a sinistra o a destra; ricevi +1 se la prendi e -1 se la manchi. Puoi aumentare la difficoltà gradualmente per trovare esattamente il punto in cui il tuo algoritmo comincia a fallire
- **Visualizza tutto**: curve di addestramento, comportamento della politica, previsioni del critico. Una tendenza che non si vede in una tabella di numeri salta all'occhio in un grafico
- **Diffida del disegno delle ricompense**: anche se in teoria la ricompensa fa parte della definizione del problema, in pratica viene regolata di continuo. Il rischio reale sta nel sovraspecificare ricompense intermedie: in un compito robotico di raggiungere, spostare e rilasciare un oggetto, premiare troppo i passi intermedi può insegnare al robot a raggiungere e spostare l'oggetto, ma mai a rilasciarlo, perché tenerlo in mano è già per lui la politica più stabile e meglio ricompensata
- **Tratta il fattore di sconto come un iperparametro in più**: anche se in teoria l'obiettivo finale non dipende da esso, scontare riduce la varianza delle stime di ritorno e può adattarsi meglio al bias induttivo del tuo ottimizzatore
- **Regola con cura la dimensione della rete**: parti piccolo perché si addestra più in fretta, ma prova ogni tanto reti più grandi. Con reti più grandi la rete target risulta meno critica, perché una rete grande confonde meno stati simili tra loro
- **Non inizializzare la politica in modo deterministico**: fin dal primo passo deve campionare azioni con una certa probabilità, non ripetere sempre la stessa
- **Resta aggiornato sull'apprendimento supervisionato**: RMSprop e Adam sono gli ottimizzatori di riferimento nell'RL profondo; tecniche come il *dropout* meritano di essere provate, anche se la normalizzazione per lotti non è ancora molto diffusa in questo campo

**Applica subito il primo consiglio!** Il codice che eseguirai tra poco costruisce esattamente quel problema giocattolo: "Catch".


## 4. IMPALA: da un computer a un cluster, con V-trace

*Traccia della lezione: disegna alla lavagna due macchine che inviano gradienti a un server centrale, con frecce di tempi diversi, così si vede fisicamente perché un gradiente può arrivare "vecchio".*

A3C risolve il problema all'interno di una sola macchina, ma la sua architettura non cresce bene oltre questo limite: se ripartisci semplicemente più apprendisti —ciascuno con una propria copia dei pesi— su macchine diverse e chiedi loro di sincronizzare i gradienti con un server centrale, emerge un problema nuovo. Quando il gradiente di una macchina arriva al server, altre macchine hanno già aggiornato i pesi, quindi quel gradiente arriva in ritardo e finisce per spingere la rete in una direzione che non corrisponde più allo stato attuale dei pesi.

### La soluzione: centralizza l'apprendimento, distribuisci l'azione

La soluzione di IMPALA (dall'inglese Importance Weighted Actor-Learner Architecture) consiste nel cambiare cosa si distribuisce. Invece di distribuire gli apprendisti, centralizza l'apprendimento in una sola copia —di solito su GPU— e distribuisci solo la parte di azione: decine o centinaia di attori su CPU percorrono l'ambiente con la politica che hanno a disposizione e inviano la loro esperienza all'apprendista centrale.

Questo evita il problema del gradiente obsoleto, ma lascia un problema gemello: quando l'esperienza di un attore arriva all'apprendista, i pesi di quest'ultimo potrebbero essere già cambiati, quindi quell'esperienza è stata generata da una politica leggermente diversa da quella attuale.

### V-trace: la correzione che dà il nome all'architettura

Qui entra V-trace, il metodo di correzione fuori politica. Usa il campionamento per importanza per ripesare ogni aggiornamento in base alla differenza tra la politica di comportamento μ dell'attore che ha generato i dati e la politica obiettivo π dell'apprendista nel momento in cui li usa.

- Se le due coincidono, l'aggiornamento di V-trace si riduce esattamente all'aggiornamento di A3C
- Quanto più si separano, tanto più peso perde quell'esperienza nell'addestramento

È un compromesso esplicito tra bias e varianza: V-trace introduce un po' di bias in cambio di una riduzione della varianza e di un addestramento stabile, anche con dati leggermente obsoleti.

### Le cifre: velocità e, soprattutto, trasferimento

*Traccia della lezione: separa chiaramente le due famiglie di cifre. Quella di velocità è quella che la classe si aspetta; quella di trasferimento tra compiti è quella che giustifica davvero la riprogettazione.*

In ambienti tridimensionali, la velocità di generazione dei dati scala così:

- A3C su una sola macchina, con 64 CPU e 32 thread: tra 6.000 e 9.000 fotogrammi al secondo
- A3C distribuito con 200 lavoratori: fino a circa 50.000 fotogrammi al secondo
- IMPALA: tra 200.000 e 250.000 fotogrammi al secondo

Ma la cifra più interessante non riguarda la velocità, bensì il trasferimento tra compiti. Nell'insieme di 30 compiti cognitivi di DeepMind Lab (DMLab-30), addestrare una rete separatamente su ogni compito e addestrarla su tutti e 30 insieme sono esperimenti diversi:

- Con A3C e una rete profonda, la prestazione media si aggira attorno al 30% del livello umano
- IMPALA con una rete piccola supera già quella cifra
- IMPALA con una rete profonda addestrata gioco per gioco migliora ulteriormente
- Il risultato migliore di tutti è IMPALA con una rete residua profonda addestrata su tutti i 30 compiti contemporaneamente con un'unica rete, al di sopra persino della versione addestrata separatamente gioco per gioco

Questo è il segnale di **trasferimento positivo**: imparare più compiti insieme, con l'architettura giusta, fa sì che ciascun compito si impari meglio e con meno dati, non peggio. In tempo di addestramento, IMPALA con rete profonda raggiunge il livello di A3C in meno di un giorno; A3C ha bisogno di circa una settimana per arrivarci.

Nell'insieme di 57 giochi di Atari, la distanza si ripete: A3C distribuito, addestrato separatamente su ogni gioco, raggiunge il 54% di punteggio medio con rete piccola e il 118% con rete profonda; IMPALA supera A3C in entrambe le dimensioni di rete, e con una rete residua profonda addestrata separatamente si avvicina al 200% di punteggio medio. Il dato più impressionante, però, è un altro: **IMPALA addestrato su tutti i 57 giochi insieme, con un'unica rete, raggiunge circa il 60% di punteggio medio** —paragonabile alla DQN originale addestrata gioco per gioco— nonostante la diversità visiva e di meccaniche tra 57 giochi di Atari sia molto maggiore di quella tra i 30 compiti di DeepMind Lab.

### Una domanda aperta

Se IMPALA è più veloce e trasferisce meglio tra i compiti, perché DQN resta in alcuni aspetti più stabile di A3C? La risposta punta all'addestramento a lotti: campionare un lotto casuale dalla memoria di replay è di per sé un ingrediente stabilizzante importante, e anche piccole deviazioni dalla politica attuale influiscono sull'actor-critic. V-trace compensa gran parte di questo effetto, ma se l'esperienza arriva troppo obsoleta, i pesi di importanza si avvicinano a zero o vengono limitati a uno, e quell'esperienza smette di fornire un segnale di apprendimento utile.


## 5. Codice dal vivo: "Catch" con un actor-critic semplice

*Traccia della lezione: prima di toccare una sola riga di codice, di' ad alta voce: "quello che costruiamo ora NON è A3C né IMPALA". A3C richiede più processi che giocano in parallelo e aggiornano una rete condivisa in modo asincrono; IMPALA richiede in più attori e apprendista distribuiti su macchine diverse, con correzione V-trace tra loro. Niente di tutto ciò è realistico dentro un singolo notebook di Colab.*

Quello che invece puoi fare —ed è esattamente ciò che raccomanda il consiglio "debugga prima in piccolo" prima di fare il salto verso il parallelismo— è costruire il banco di prova "Catch" e addestrarvi sopra un actor-critic semplice a processo singolo, con la stessa idea che hai visto nella lezione nove. Serve a verificare che la logica dell'algoritmo funzioni prima di complicare l'architettura.

Ti servono solo `numpy`, `torch` e `matplotlib`, tutte e tre preinstallate in Google Colab. Lo costruirai cella per cella:

1. Importiamo le librerie
2. Definiamo l'ambiente "Catch"
3. Definiamo la rete actor-critic
4. Scriviamo la funzione che gioca un episodio completo
5. Scriviamo la funzione che aggiorna la rete
6. Addestriamo e disegniamo il tasso di successo

**Esegui ogni cella in ordine.**


In [ ]:
# --- 1. Importamos las librerias ---
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


### L'ambiente "Catch"

*Traccia della lezione: disegna la griglia alla lavagna —dieci righe per cinque colonne— prima di mostrare il codice alla classe. Segna dove compare la palla e dove vive la racchetta, e chiarisci che la racchetta misura due colonne di larghezza.*

La palla compare in una colonna casuale della riga superiore e scende di una riga a ogni passo. Controlli una racchetta larga due pixel nella riga inferiore, che puoi spostare a sinistra, a destra o lasciare ferma. Quando la palla raggiunge la riga inferiore, ricevi +1 se la racchetta la copre e -1 se no.


In [ ]:
# --- 2. El entorno "Catch": una pelota cae, una pala debe atraparla ---
FILAS = 10
COLUMNAS = 5
ANCHO_PALA = 2
N_ACCIONES = 3  # 0 = izquierda, 1 = quieta, 2 = derecha


class Catch:
    """Implementacion minima del juego de juguete "Catch"."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(COLUMNAS)
        # Punto de partida aleatorio de la pala, sin salirse del borde
        self.columna_pala = np.random.randint(COLUMNAS - ANCHO_PALA + 1)
        return self._estado()

    def _estado(self):
        """Codifica el estado como una imagen binaria aplanada: un uno
        donde esta la pelota y unos donde ocupa la pala."""
        grid = np.zeros((FILAS, COLUMNAS), dtype=np.float32)
        grid[self.fila_pelota, self.columna_pelota] = 1.0
        grid[FILAS - 1, self.columna_pala:self.columna_pala + ANCHO_PALA] = 1.0
        return grid.flatten()

    def step(self, accion):
        # Mueve la pala; np.clip evita que se salga de la cuadricula
        desplazamiento = {0: -1, 1: 0, 2: 1}[accion]
        self.columna_pala = int(
            np.clip(self.columna_pala + desplazamiento, 0, COLUMNAS - ANCHO_PALA)
        )
        # La pelota cae una fila en cada paso
        self.fila_pelota += 1
        terminado = self.fila_pelota == FILAS - 1
        if terminado:
            atrapada = (
                self.columna_pala
                <= self.columna_pelota
                < self.columna_pala + ANCHO_PALA
            )
            recompensa = 1.0 if atrapada else -1.0
        else:
            recompensa = 0.0
        return self._estado(), recompensa, terminado


### La rete actor-critic

*Traccia della lezione: fai notare che c'è una sola base condivisa e due teste di output. Chiedi alla classe perché condividere la base ha senso qui (politica e valore partono dalla stessa informazione sullo stato).*

Una base condivisa elabora lo stato; una testa produce la politica (probabilità di ogni azione) e un'altra il valore dello stato, come hai visto nella lezione nove.


In [ ]:
# --- 3. Red actor-critico: una base compartida y dos cabezas de salida ---
class ActorCritico(nn.Module):
    """Red pequena con una base compartida: una cabeza produce la
    politica (probabilidad de cada accion) y otra el valor del estado."""

    def __init__(self, dim_entrada, n_acciones, dim_oculta=128):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(dim_entrada, dim_oculta),
            nn.ReLU(),
        )
        self.cabeza_politica = nn.Linear(dim_oculta, n_acciones)
        self.cabeza_valor = nn.Linear(dim_oculta, 1)

    def forward(self, x):
        h = self.base(x)
        return self.cabeza_politica(h), self.cabeza_valor(h)


### Iperparametri e semi

Fissiamo il seme casuale perché l'esperimento sia riproducibile. `COEF_ENTROPIA` è il consiglio pratico "non inizializzare la politica in modo deterministico" trasformato in iperparametro: penalizza il fatto che la politica diventi troppo sicura di sé troppo presto.


In [ ]:
# --- 4. Hiperparametros, semillas y objetos del experimento ---
GAMMA = 0.99
TASA_APRENDIZAJE = 1e-3
N_EPISODIOS = 3000
COEF_VALOR = 0.5
COEF_ENTROPIA = 0.01  # fomenta la exploracion: la politica nunca debe
                       # arrancar (ni quedarse) deterministica

torch.manual_seed(0)
np.random.seed(0)

entorno = Catch()
red = ActorCritico(dim_entrada=FILAS * COLUMNAS, n_acciones=N_ACCIONES)
optimizador = torch.optim.Adam(red.parameters(), lr=TASA_APRENDIZAJE)


### Giocare un episodio completo

*Traccia della lezione: percorri il ciclo `while` riga per riga. Insisti sul fatto che `distribucion.sample()` è la politica che campiona un'azione, non che sceglie sempre la migliore.*

Questa funzione gioca a "Catch" dall'inizio alla fine con la politica attuale e salva, passo dopo passo, tutto ciò che serve per aggiornare la rete alla fine: il logaritmo della probabilità dell'azione scelta, il valore stimato dello stato, la ricompensa ottenuta e l'entropia della politica.


In [ ]:
# --- 5. Un episodio completo con la politica actual ---
def ejecutar_episodio():
    """Juega un episodio completo de Catch y devuelve, paso a paso, el
    log-probabilidad de la accion elegida, el valor estimado del
    estado, la recompensa obtenida y la entropia de la politica."""
    estado = entorno.reset()
    log_probs, valores, recompensas, entropias = [], [], [], []
    terminado = False
    while not terminado:
        entrada = torch.from_numpy(estado)
        logits, valor = red(entrada)
        distribucion = torch.distributions.Categorical(logits=logits)
        accion = distribucion.sample()

        estado, recompensa, terminado = entorno.step(accion.item())

        log_probs.append(distribucion.log_prob(accion))
        entropias.append(distribucion.entropy())
        valores.append(valor.squeeze())
        recompensas.append(recompensa)

    return log_probs, valores, recompensas, entropias


### Aggiornare l'actor-critic

*Traccia della lezione: fermati sulla riga del `ventaja` e chiedi cosa succederebbe se non si usasse `valores.detach()`. È lo stesso concetto di vantaggio che hai già studiato nella lezione nove, ora dentro un ciclo di addestramento completo.*

Calcola il ritorno scontato a ogni passo, il vantaggio rispetto al valore stimato dal critico, e aggiorna attore e critico insieme con un'unica perdita combinata.


In [ ]:
# --- 6. Actualizacion del actor-critico con un episodio completo ---
def actualizar_red(log_probs, valores, recompensas, entropias):
    """Calcula el retorno descontado en cada paso, la ventaja frente al
    valor estimado por el critico, y actualiza actor y critico a la vez."""
    retornos = []
    G = 0.0
    for r in reversed(recompensas):
        G = r + GAMMA * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos, dtype=torch.float32)

    valores = torch.stack(valores)
    ventaja = retornos - valores.detach()  # sin gradiente hacia el critico aqui

    perdida_actor = -(torch.stack(log_probs) * ventaja).mean()
    perdida_critico = F.mse_loss(valores, retornos)
    perdida_entropia = torch.stack(entropias).mean()

    perdida = (
        perdida_actor
        + COEF_VALOR * perdida_critico
        - COEF_ENTROPIA * perdida_entropia
    )

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


### Addestriamo e misuriamo il tasso di successo

*Traccia della lezione: prima di eseguire, chiedi alla classe quale tasso di successo si aspetta all'inizio. Con tre azioni e due posizioni della racchetta in una griglia di cinque colonne, il punto di partenza è attorno al 40%, vicino al caso. La cella impiega alcuni secondi: approfitta per spiegare cosa succede episodio dopo episodio mentre gira.*

**Esegui la cella: addestra per 3.000 episodi completi.**


In [ ]:
# --- 7. Bucle de entrenamiento: 3000 episodios completos ---
aciertos = []  # 1 si el episodio termino en captura, 0 si termino en fallo

for episodio in range(N_EPISODIOS):
    log_probs, valores, recompensas, entropias = ejecutar_episodio()
    actualizar_red(log_probs, valores, recompensas, entropias)
    aciertos.append(1 if recompensas[-1] > 0 else 0)

print("Entrenamiento terminado:", N_EPISODIOS, "episodios")


**Esegui questa cella per disegnare il tasso di successo.** Usiamo una media mobile di 100 episodi per attenuare il rumore da episodio a episodio e far vedere la tendenza reale di apprendimento.


In [ ]:
# --- 8. Graficamos la tasa de aciertos con una media movil ---
VENTANA = 100
tasa_movil = np.convolve(aciertos, np.ones(VENTANA) / VENTANA, mode="valid")

plt.figure(figsize=(8, 5))
plt.plot(tasa_movil)
plt.xlabel("Episodios")
plt.ylabel(f"Tasa de aciertos (media movil de {VENTANA} episodios)")
plt.title('Actor-critico simple aprendiendo a jugar "Catch"')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()


**Lettura del risultato**

*Traccia della lezione: fermati su questo grafico e chiedi alla classe se la forma della curva somiglia a qualcuna già vista nelle lezioni precedenti (per esempio, nella lezione nove con Dyna-Q).*

Vedrai il tasso di successo partire da vicino al caso —attorno al 40%— e salire in modo costante fino ad avvicinarsi a catture quasi sempre riuscite, man mano che la rete impara ad allineare la racchetta alla colonna della palla. Se la curva si blocca, prova ad aumentare `COEF_ENTROPIA` (la politica smette di esplorare troppo presto) oppure a diminuire `TASA_APRENDIZAJE` (gli aggiornamenti sono troppo bruschi per un problema così piccolo).


## Punti chiave

- DQN risolve la correlazione dei suoi dati con una memoria di replay e una rete target, ma questo lo lega all'apprendimento fuori politica e spreca la GPU
- A3C sostituisce la memoria con il parallelismo: molti lavoratori, ciascuno con una propria copia dell'ambiente, aggiornano in modo asincrono una rete condivisa senza bloccarsi a vicenda
- La diversità simultanea tra i lavoratori spezza la correlazione dei dati come prima faceva la diversità temporale della memoria di replay, senza bisogno di memorizzare nulla
- La componente che impara dentro ogni lavoratore è un actor-critic di vantaggio: una politica normalizzata (che somma a uno) più un critico che fa da riferimento
- IMPALA centralizza l'apprendimento in una sola copia e distribuisce solo l'azione tra molti attori, e usa V-trace per correggere lo sfasamento tra la politica che ha generato i dati e la politica attuale dell'apprendista
- IMPALA non è solo più veloce: addestrare più compiti insieme con l'architettura giusta produce trasferimento positivo, migliorando la prestazione su ciascun compito
- Il gioco giocattolo "Catch" è il banco di prova consigliato per debuggare la logica di un agente prima di complicare l'architettura con il parallelismo
- Il disegno delle ricompense, il fattore di sconto, la dimensione della rete e l'inizializzazione non deterministica della politica sono leve pratiche importanti quanto l'algoritmo scelto

## Prima della prossima lezione

**Apri il notebook di pratica di questa lezione e completa i due esercizi su "Catch".** Programmerai tu stesso da zero la funzione di passo dell'ambiente e misurerai l'effetto reale di cambiare la dimensione della griglia e la velocità di caduta della palla.

Nella lezione 12, l'ultima del corso, vedrai come queste idee sono state combinate —insieme alla ricerca ad albero e all'autogioco— per costruire sistemi come AlphaGo, AlphaZero e AlphaStar.
